# TinyGPT-2: pretrain GPT-2 small on TinyStories

Trains a GPT-2 (124M parameters) written from scratch on the TinyStories dataset, on **one Colab GPU**.
Everything lives on Google Drive, so a dropped session costs you at most one checkpoint interval.

**Before you start**
1. *Runtime → Change runtime type →* pick the **A100 GPU** (availability depends on your Colab plan). bf16 mixed precision is used automatically.
2. Fill in `REPO_URL` in the settings cell below. For a private repo, add a Colab secret named `GITHUB_TOKEN` (key icon in the left bar) or paste a token when asked.
3. **Do a dry run first:** set `SMOKE_TEST = True`, *Run all*, and check that it finishes in a couple of minutes. It uses the small 20M model, a tiny slice of data, and its own folders, so it cannot disturb the real run.
4. Then set `SMOKE_TEST = False` and *Run all* for the real run.

**If the session drops:** reconnect and *Run all* again. Data preparation is skipped when it is already on Drive, and training resumes from the latest checkpoint.

## 1. Settings

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# ---- Settings you may change ------------------------------------------------------------
REPO_URL = "https://github.com/YOUR-USER/TinyGPT-2.git"   # <- fill in (private repos are fine)
BRANCH = "main"
DRIVE_DIR = "TinyGPT-2"            # folder inside My Drive holding data, checkpoints and reports
RUN_NAME = "pretrain-gpt2-small"   # one folder per run: change it to start a different run

SMOKE_TEST = False                 # True = 2-minute dry run on the small model (see above)

PRESET = "gpt2_small"              # "gpt2_small" (124M parameters) or "small" (~20M)
# GPT-2 small on an A100 (40 GB). The attention here is written out step by step, which
# costs memory. If you get "CUDA out of memory": halve BATCH_SIZE and double GRAD_ACCUM_STEPS.
# Their product, the effective batch, stays the same, so training is unaffected.
BATCH_SIZE = 8                     # sequences per forward/backward pass
GRAD_ACCUM_STEPS = 64              # 8 * 1024 tokens * 64 = 524,288 tokens per optimizer step
MAX_TOKENS = None                  # None = one pass over TinyStories (~470M tokens)
LEARNING_RATE = 6e-4
CHECKPOINT_INTERVAL = 100          # optimizer steps between checkpoints (~1.5 GB each on Drive)
SEED = 1337
# -------------------------------------------------------------------------------------------

# Lets the automated test run the dry run without editing this cell.
SMOKE_TEST = os.environ.get("TINYGPT_SMOKE_TEST", str(SMOKE_TEST)) == "True"

EXTRA = {}  # extra TrainConfig settings
if SMOKE_TEST:
    PRESET, BATCH_SIZE, GRAD_ACCUM_STEPS, MAX_TOKENS = "small", 4, 1, 40_960
    CHECKPOINT_INTERVAL = 10
    EXTRA = dict(warmup_steps=5, eval_interval=10, eval_batches=5, log_interval=10, sample_tokens=32)
    RUN_NAME += "-smoke"

print(f"preset={PRESET}  run={RUN_NAME}  smoke_test={SMOKE_TEST}")

## 2. Check the GPU

In [ ]:
import torch

IN_COLAB = "google.colab" in sys.modules

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name()
    print(f"GPU: {gpu}  ({torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB)")
    print("bf16 supported:", torch.cuda.is_bf16_supported())
    if "A100" not in gpu:
        print("Note: these settings are tuned for an A100. On a smaller GPU lower BATCH_SIZE "
              "(and raise GRAD_ACCUM_STEPS); MFU is only reported for known GPUs.")
else:
    print("No GPU found. " + ("Choose a GPU runtime: Runtime > Change runtime type."
                              if IN_COLAB else "Running on the CPU (fine for the dry run)."))

## 3. Google Drive and folders

Everything that must survive the session goes to Drive: prepared data, checkpoints, the Profile report.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive") / DRIVE_DIR
else:
    # Not on Colab (e.g. a local dry run): use a scratch folder instead of Drive.
    WORK = Path(os.environ.get("TINYGPT_WORKDIR", Path.cwd() / ".notebook_runs"))

# The dry run gets its own data folder so its fake "train" split can never be mistaken for real data.
DATA_DIR = WORK / ("data_smoke" if SMOKE_TEST else "data")
RUN_DIR = WORK / "runs" / RUN_NAME
CKPT_DIR = RUN_DIR / "checkpoints"
for folder in (DATA_DIR, RUN_DIR):
    folder.mkdir(parents=True, exist_ok=True)
print("work folder:", WORK)

## 4. Get the code

On Colab the repository is cloned (or updated) from GitHub; the token, if any, is never stored in the clone.

In [ ]:
if IN_COLAB:
    REPO_DIR = Path("/content/TinyGPT-2")
    if "YOUR-USER" in REPO_URL:
        raise ValueError("Set REPO_URL in the settings cell to your GitHub repository.")

    token = ""
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN") or ""
    except Exception:
        pass
    if not token:
        from getpass import getpass
        token = getpass("GitHub token (leave empty for a public repo): ").strip()
    auth_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

    def sh(cmd):
        # Run a command; if it fails, show git's message with the token masked out.
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode:
            message = (result.stderr or result.stdout)
            raise RuntimeError(message.replace(token, "***") if token else message)

    if (REPO_DIR / ".git").exists():
        sh(["git", "-C", str(REPO_DIR), "pull", "--ff-only", auth_url, BRANCH])
    else:
        sh(["git", "clone", "--branch", BRANCH, auth_url, str(REPO_DIR)])
    sh(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL])  # drop the token
else:
    # Local run: use the repository this notebook lives in.
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "model" / "gpt2.py").exists())

sys.path.insert(0, str(REPO_DIR))
print("code:", REPO_DIR)

## 5. Install dependencies

Colab already has PyTorch; only the rest is installed. (The bitsandbytes and transformers packages are needed later, for fine-tuning and weight checks, not here.)

In [ ]:
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "datasets", "tiktoken", "matplotlib", "tensorboard"], check=True)
else:
    print("Not on Colab: assuming the dependencies are installed (pip install -r requirements.txt).")

## 6. Prepare the data

Downloads TinyStories, tokenizes it with the GPT-2 tokenizer (`<|endoftext|>` after every story) and saves token shards to Drive. Skipped if the shards are already there, so re-running is cheap. The full download and tokenization takes several minutes the first time.

In [ ]:
from data.prepare import load_tinystories, prepare_split

prepare_split("validation", DATA_DIR)
if SMOKE_TEST:
    # Dry run: reuse the small validation split as the training data too.
    prepare_split("train", DATA_DIR, get_stories=lambda: load_tinystories("validation"))
else:
    prepare_split("train", DATA_DIR)

## 7. Train

Resumes automatically from the latest checkpoint in `CKPT_DIR`, so this cell is safe to re-run after a disconnect. Each progress line shows loss, learning rate, gradient norm and tokens per second; validation runs every `eval_interval` steps.

In [ ]:
from model.config import gpt2_small, small
from train.pretrain import TrainConfig, pretrain

model_config = {"gpt2_small": gpt2_small, "small": small}[PRESET]()
cfg = TrainConfig(
    data_dir=str(DATA_DIR),
    out_dir=str(RUN_DIR),
    checkpoint_dir=str(CKPT_DIR),
    batch_size=BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM_STEPS,
    max_tokens=MAX_TOKENS,
    learning_rate=LEARNING_RATE,
    checkpoint_interval=CHECKPOINT_INTERVAL,
    seed=SEED,
    **EXTRA,
)
result = pretrain(model_config, cfg)

## 8. Profile report

How the run went: loss and perplexity, speed, memory, MFU (how much of the GPU's peak arithmetic speed the model uses), and what the model writes.

In [ ]:
import json
from IPython.display import Image, display

report = result.report
print(json.dumps(report["summary"], indent=2))
for note in report["notes"]:
    print("NOTE:", note)

display(Image(filename=report["files"]["loss_curves"]))

print("\nLatest sample generations:")
for sample in report["samples"][-len(cfg.sample_prompts):]:
    print(f"\n--- step {sample['step']} | prompt: {sample['prompt']!r}\n{sample['text']}")

## Where everything is

All in `My Drive/<DRIVE_DIR>/runs/<RUN_NAME>/`:

| File | What |
|---|---|
| `profile_report.json` | all the numbers above, plus every sample generation |
| `loss_curves.png` | train and validation loss, validation perplexity |
| `profile/*.json` | `torch.profiler` trace of a few steps: open it at [ui.perfetto.dev](https://ui.perfetto.dev) or `chrome://tracing`, or in TensorBoard with `pip install torch_tb_profiler` |
| `config.json` | the complete configuration of the run |
| `checkpoints/` | the newest checkpoints; the next stage (fine-tuning) starts from the latest one |

If the run was cut short, *Run all* again to continue from the last checkpoint.